cài systemds

In [ ]:
!pip install systemds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 MB 6.9 MB/s eta 0:00:00


test

In [ ]:
from systemds.context import SystemDSContext
import numpy as np

with SystemDSContext() as sds:
    m = sds.from_numpy(np.array([[1,2],[3,4]]))
    print(m.compute())

[[1 2]
 [3 4]]


DML

In [ ]:
import numpy as np

X = np.array([[1,2],
              [3,4]])

np.savetxt("X.csv", X, delimiter=",")

tạo file dml

In [ ]:
dml_script = """
demo = function(Matrix[Double] X) return (Matrix[Double] XtX) {
    XtX = t(X) %*% X;
}
"""

with open("demo.dml", "w") as f:
    f.write(dml_script)

print("Đã tạo file demo.dml")

Đã tạo file demo.dml


chạy dml bằng systemds

In [ ]:
from systemds.context import SystemDSContext

with SystemDSContext() as sds:
    pkg = sds.source("demo.dml", "demo_pkg")
    XtX = pkg.demo(sds.from_numpy(X)).compute()
    print(XtX)

[[10. 14.]
 [14. 20.]]


Nhận xét về phần demo DML

Trong phần demo này, nhóm đã sử dụng ngôn ngữ DML để xây dựng và thực thi một phép toán ma trận cơ bản, từ đó mở rộng sang mô tả thuật toán Logistic Regression ở mức tối giản. Qua đó có thể thấy rõ bản chất của DML không phải là một ngôn ngữ lập trình thông thường như Python, mà là một ngôn ngữ mang tính khai báo, tập trung vào việc mô tả các phép toán và công thức toán học.

Cụ thể, nhóm không sử dụng các thư viện machine learning có sẵn như scikit-learn, mà trực tiếp biểu diễn các bước của thuật toán Logistic Regression thông qua các phép toán ma trận như nhân ma trận, chuyển vị và hàm sigmoid. Điều này cho thấy DML cho phép người dùng làm việc ở mức trừu tượng cao, gần với công thức toán học hơn là chi tiết triển khai.

Bên cạnh đó, quá trình thực thi không diễn ra trực tiếp trong Python. Python chỉ đóng vai trò trung gian để truyền dữ liệu và gọi hàm, trong khi SystemDS (SystemML) mới là thành phần tiếp nhận đoạn mã DML, thực hiện phân tích, tối ưu và lựa chọn cách thực thi phù hợp. Điều này thể hiện rõ sự tách biệt giữa việc mô tả thuật toán và việc thực thi thuật toán.

Từ demo, có thể kết luận rằng DML đóng vai trò như một lớp mô tả logic tính toán, còn SystemML/SystemDS đóng vai trò như một trình biên dịch và engine thực thi, giúp tối ưu hóa quá trình xử lý dữ liệu và mô hình học máy.

Thực thi LR bằng dml trên colab

b1. tạo dml

In [ ]:
dml_script = """
lr = function(Matrix[Double] X, Matrix[Double] y) return (Matrix[Double] w) {

    w = matrix(0, rows=ncol(X), cols=1)

    for(i in 1:10) {
        z = X %*% w
        p = 1 / (1 + exp(-z))
        grad = t(X) %*% (p - y)
        w = w - 0.01 * grad
    }
}
"""

with open("lr.dml", "w") as f:
    f.write(dml_script)

b2. data

In [ ]:
import numpy as np

X = np.array([[1, 2],
              [3, 4]], dtype=float)

y = np.array([[0],
              [1]], dtype=float)

verbose

In [ ]:
from systemds.context import SystemDSContext

with SystemDSContext() as sds:
    pkg = sds.source("lr.dml", "lr_pkg")
    w = pkg.lr(
        sds.from_numpy(X),
        sds.from_numpy(y)
    ).compute(verbose=True)

    print("Final w:", w)

SCRIPT:
source("lr.dml") as lr_pkg
V1=read('./tmp/V1',rows=2,cols=2);
V2=read('./tmp/V2',rows=2,cols=1);
V3=lr_pkg::lr(X=V1,y=V2);
write(V3, './tmp');

Final w: [[0.07792073]
 [0.0687175 ]]


trong ví dụ trên: mình đã thực hiện việc tính toán lr trên systemds nhưng kết quả khá mơ hồ (tức là chúng ta chưa thể có cái nhìn tổng quát về systemmml có gì khác biệt so với các thư viện py thông thường),

Vậy nên bước tiếp theo mình sẽ tính toán trên numpy và systemml để xem sự khác biệt là gì

In [ ]:
import numpy as np

# Dữ liệu
X = np.array([[1, 2],
              [3, 4]], dtype=float)

y = np.array([[0],
              [1]], dtype=float)

# Khởi tạo
w = np.zeros((X.shape[1], 1))
lr = 0.01
epochs = 10

print("w ban đầu =\n", w)

for i in range(epochs):
    z = X @ w
    p = 1 / (1 + np.exp(-z))
    grad = X.T @ (p - y)
    w = w - lr * grad

    print(f"\nVòng lặp {i+1}")
    print("z =\n", z)
    print("p =\n", p)
    print("grad =\n", grad)
    print("w =\n", w)

print("\nFinal w =\n", w)

w ban đầu =
 [[0.]
 [0.]]

Vòng lặp 1
z =
 [[0.]
 [0.]]
p =
 [[0.5]
 [0.5]]
grad =
 [[-1.]
 [-1.]]
w =
 [[0.01]
 [0.01]]

Vòng lặp 2
z =
 [[0.03]
 [0.07]]
p =
 [[0.50749944]
 [0.51749286]]
grad =
 [[-0.94002199]
 [-0.91502969]]
w =
 [[0.01940022]
 [0.0191503 ]]

Vòng lặp 3
z =
 [[0.05770081]
 [0.13480185]]
p =
 [[0.5144212 ]
 [0.53364952]]
grad =
 [[-0.88463023]
 [-0.83655951]]
w =
 [[0.02824652]
 [0.02751589]]

Vòng lặp 4
z =
 [[0.08327831]
 [0.19480313]]
p =
 [[0.52080755]
 [0.54854736]]
grad =
 [[-0.83355038]
 [-0.76419547]]
w =
 [[0.03658203]
 [0.03515785]]

Vòng lặp 5
z =
 [[0.10689772]
 [0.25037746]]
p =
 [[0.52669901]
 [0.56226941]]
grad =
 [[-0.78649277]
 [-0.69752436]]
w =
 [[0.04444695]
 [0.04213309]]

Vòng lặp 6
z =
 [[0.12871313]
 [0.30187322]]
p =
 [[0.53213393]
 [0.57490038]]
grad =
 [[-0.74316494]
 [-0.63613063]]
w =
 [[0.0518786]
 [0.0484944]]

Vòng lặp 7
z =
 [[0.1488674]
 [0.3496134]]
p =
 [[0.53714827]
 [0.58652383]]
grad =
 [[-0.70328026]
 [-0.57960816]]
w =
 [[0.05

DML script

   ↓


Parser

   ↓


HOP graph (hiểu bài toán)

   ↓

Rewrite / Optimization

   ↓

LOP (kế hoạch chạy)

   ↓

Runtime plan (local / Spark)


   ↓

Execution

   ↓
   
Kết quả

demo credit card
phần 1: py
1. load data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("creditcard.csv")

print("Shape ban đầu:", df.shape)
print("Tổng số NaN:", df.isna().sum().sum())
print("\nNaN theo cột:")
print(df.isna().sum()[df.isna().sum() > 0])

Shape ban đầu: (284807, 31)
Tổng số NaN: 0

NaN theo cột:
Series([], dtype: int64)


2. xem phân bố nhãn

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

X shape: (284807, 30)
y shape: (284807,)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64


3. xử lý nan

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

print("Shape sau impute:", X_imputed.shape)
print("Số NaN còn lại:", np.isnan(X_imputed).sum())

Shape sau impute: (284807, 30)
Số NaN còn lại: 0


4. scale

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Shape sau scale:", X_scaled.shape)
print("Số NaN sau scale:", np.isnan(X_scaled).sum())

Shape sau scale: (284807, 30)
Số NaN sau scale: 0


4. baseline Python

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=100, random_state=42)
model.fit(X_scaled, y)

y_pred = model.predict(X_scaled)

print("Accuracy:", accuracy_score(y, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y, y_pred))
print("\nClassification report:")
print(classification_report(y, y_pred, digits=4))

Accuracy: 0.9991924355791817

Confusion matrix:
[[284274     41]
 [   189    303]]

Classification report:
              precision    recall  f1-score   support

           0     0.9993    0.9999    0.9996    284315
           1     0.8808    0.6159    0.7249       492

    accuracy                         0.9992    284807
   macro avg     0.9401    0.8079    0.8622    284807
weighted avg     0.9991    0.9992    0.9991    284807



lưu data cho demo systemml

In [ ]:
np.savetxt("X_credit.csv", X_scaled, delimiter=",")
np.savetxt("y_credit.csv", y.values.reshape(-1, 1), delimiter=",")

print("Saved X_credit.csv:", X_scaled.shape)
print("Saved y_credit.csv:", y.values.reshape(-1, 1).shape)

Saved X_credit.csv: (284807, 30)
Saved y_credit.csv: (284807, 1)


phần 2: systemml

1. tạo .dml

In [ ]:
dml_script = """
lr = function(Matrix[Double] X, Matrix[Double] y) return (Matrix[Double] w) {

    n = nrow(X);
    m = ncol(X);

    w = matrix(0, rows=m, cols=1);

    for(i in 1:10) {
        z = X %*% w;
        p = 1 / (1 + exp(-z));
        grad = t(X) %*% (p - y) / n;
        w = w - 0.1 * grad;
    }
}
"""

with open("credit_lr.dml", "w") as f:
    f.write(dml_script)

print("Đã tạo file credit_lr.dml")

Đã tạo file credit_lr.dml


2. run

In [ ]:
from systemds.context import SystemDSContext
import numpy as np
import time

start = time.time()

with SystemDSContext() as sds:
    pkg = sds.source("credit_lr.dml", "credit_pkg")
    w = pkg.lr(
        sds.from_numpy(X_scaled),
        sds.from_numpy(y.values.reshape(-1, 1))
    ).compute(verbose=True)

end = time.time()

print("Thời gian chạy SystemDS:", round(end - start, 2), "giây")
print("Shape của w:", w.shape)
print("5 giá trị đầu của w:\n", w[:5])

SCRIPT:
source("credit_lr.dml") as credit_pkg
V1=read('./tmp/V1',rows=284807,cols=30);
V2=read('./tmp/V2',rows=284807,cols=1);
V3=credit_pkg::lr(X=V1,y=V2);
write(V3, './tmp');

Thời gian chạy SystemDS: 7.79 giây
Shape của w: (30, 1)
5 giá trị đầu của w:
 [[-0.00042209]
 [-0.00377042]
 [ 0.00341593]
 [-0.00721114]
 [ 0.00496473]]


| Tiêu chí           | NumPy     | SystemDS         |
| ------------------ | --------- | ---------------- |
| Thực thi           | trực tiếp | qua DML + engine |
| Có compiler        | ❌         | ✅                |
| Có execution layer | ❌         | ✅                |
